# <center> Homework 138

In [1]:
import tensorflow as tf
# import tf_model
# import tf_data
from importlib import reload
import tensorflow_datasets as tfds
from pathlib import Path
import numpy as np
import pandas as pd

/usr/local/lib/python3.12/dist-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
print(tf.__version__)
print(tf.keras.__version__)

2.20.0
3.13.0


## Task 1
да се имплементира класа Attention

    парметъра score_mode да поддъража dot и concat


In [3]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets/spa-eng", extract=True)

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
text = (Path('/tmp/.keras/datasets/spa-eng_extracted/spa-eng').with_name("spa-eng") / "spa.txt").read_text()
# text = (Path('datasets/spa-eng/datasets/spa-eng_extracted/spa-eng').with_name("spa-eng") / "spa.txt").read_text()

In [5]:
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs)

In [6]:
vocab_size = 1000
max_length = 50

text_vec_layer_en = tf.keras.layers.TextVectorization(
                        vocab_size, output_sequence_length=max_length)

text_vec_layer_es = tf.keras.layers.TextVectorization(
                        vocab_size, output_sequence_length=max_length)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

In [7]:
X_train = tf.constant(sentences_en[:100_000])
X_valid = tf.constant(sentences_en[100_000:])

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])

Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])

![Attention_layer](Attention_layer.png)

In [ ]:
class Attention(tf.keras.layers.Layer):
    """
    Dot product attention → Luong-style
    Concat / additive attention → Bahdanau-style
    """
    def __init__(self,
                 use_scale=False,
                 score_mode='dot',
                 **kwargs):

        super().__init__(**kwargs)
        self.use_scale  = use_scale
        self.score_mode = score_mode
        self.support_masking = True

        if score_mode == 'concat':
            self.dense = tf.keras.layers.Dense(1)

    def call(self, inputs, mask=None, training=False): # [decoder_outputs: query, encoder_outputs: value, ? :key] default value = key
        if len(inputs) == 2:
            query, value = inputs
            key = value
        elif len(inputs) == 3:
            query, value, key = inputs
        else:
            raise ValueError('Invalid call inputs; expect [query, value, key] or [query, (value, key)]')

        if self.score_mode == 'dot':
            score = tf.matmul(query, key, transpose_b=True)

        elif self.score_mode == 'concat':
            query_ext = tf.expand_dims(query, axis=2)
            key_ext   = tf.expand_dims(key, axis=1)

            query_ext = tf.tile(query_ext, [1, 1, key_ext.shape[2], 1])
            key_ext   = tf.tile(key_ext, [1, query_ext.shape[1], 1, 1])

            concat = tf.concat([query_ext, key_ext], axis=-1)

            score = self.dense(tf.nn.tanh(concat))
            score = tf.squeeze(score, axis=-1)

        else:
            raise NotImplemented()

        if self.use_scale:
            f = tf.cast(tf.shape(query)[-1], tf.float32)
            score /= tf.sqrt(f)

        if mask is not None:
            if isinstance(mask, (list, tuple)):
                mask = mask[0]

            mask = tf.cast(mask, score.dtype)
            mask = mask[:, tf.newaxis, :]
            score += (1.0 - mask) * -1e9 # -1e9 = -1 000 000 000 -> softmax -> 0

        softmax = tf.nn.softmax(score, axis=-1)
        context = tf.matmul(softmax, value)

        return context

In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True, return_state=True))
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

encoder_state = [tf.keras.layers.concatenate(encoder_state[::2], axis=-1),  # short-term (0 & 2)
                 tf.keras.layers.concatenate(encoder_state[1::2], axis=-1)] # long-term (1 & 3)

# DECODER
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

# Attention
attention_layer = Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'attention' (of type Attention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [ ]:
nmt_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

nmt_model.fit((X_train, X_train_dec), Y_train,
              epochs=10,
              validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 24ms/step - accuracy: 0.8849 - loss: 0.8641 - val_accuracy: 0.9089 - val_loss: 0.4977
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 75s 24ms/step - accuracy: 0.9117 - loss: 0.4719 - val_accuracy: 0.9194 - val_loss: 0.4015
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 75s 24ms/step - accuracy: 0.9224 - loss: 0.3760 - val_accuracy: 0.9291 - val_loss: 0.3269
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 75s 24ms/step - accuracy: 0.9319 - loss: 0.3054 - val_accuracy: 0.9367 - val_loss: 0.2768
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 75s 24ms/step - accuracy: 0.9399 - loss: 0.2565 - val_accuracy: 0.9410 - val_loss: 0.2524
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 81s 26ms/step - accuracy: 0.9455 - loss: 0.2266 - val_accuracy: 0.9448 - val_loss: 0.2337
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 75s 24ms/step - accuracy: 0.9486 - loss: 0.2098 - val_accuracy: 0.9467 - val_loss: 0.2236
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 76s 24ms/step - accuracy: 0.9517 -

In [12]:
def translate(model, sentence_en):
    translation = ""
    for word_idx in range(max_length):
        X = tf.constant([sentence_en])  # encoder input
        X_dec = tf.constant(["startofseq " + translation])  # decoder input
        y_proba = model.predict((X, X_dec))[0, word_idx]  # last token's probas
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()

In [ ]:
translate(nmt_model, "I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'me gusta jugar y y va a la playa'

## Task 2
да се имплементира класа AdditiveAttention

In [8]:
class AdditiveAttention(tf.keras.layers.Layer):
    """
    Dot product attention → Luong-style
    Concat / additive attention → Bahdanau-style
    """
    def __init__(self, use_scale=True, **kwargs):
        super().__init__(**kwargs)

        self.use_scale  = use_scale
        self.dense = tf.keras.layers.Dense(1)
        self.support_masking = True

    def call(self, inputs, mask=None, training=False): # [decoder_outputs: query, encoder_outputs: value, ? :key] default value = key
        if len(inputs) == 2:
            query, value = inputs
            key = value
        elif len(inputs) == 3:
            query, value, key = inputs
        else:
            raise ValueError('Invalid call inputs; expect [query, value, key] or [query, (value, key)]')

        query_ext = tf.expand_dims(query, axis=2)
        key_ext   = tf.expand_dims(key, axis=1)

        query_ext = tf.tile(query_ext, [1, 1, key_ext.shape[2], 1])
        key_ext   = tf.tile(key_ext, [1, query_ext.shape[1], 1, 1])

        concat = tf.concat([query_ext, key_ext], axis=-1)

        score = self.dense(tf.nn.tanh(concat))
        score = tf.squeeze(score, axis=-1)

        if mask is not None:
            if isinstance(mask, (list, tuple)):
                mask = mask[0]

            mask = tf.cast(mask, score.dtype)
            mask = mask[:, tf.newaxis, :]
            score += (1.0 - mask) * -1e9 # -1e9 = -1 000 000 000 -> softmax -> 0

        if self.use_scale:
            f = tf.cast(tf.shape(query)[-1], tf.float32)
            score /= tf.sqrt(f)

        softmax = tf.nn.softmax(score, axis=-1)
        return tf.matmul(softmax, value)

In [9]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True, return_state=True))
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

encoder_state = [tf.keras.layers.concatenate(encoder_state[::2], axis=-1),  # short-term (0 & 2)
                 tf.keras.layers.concatenate(encoder_state[1::2], axis=-1)] # long-term (1 & 3)

# DECODER
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

# AdditiveAttention
attention_layer = AdditiveAttention(use_scale=False)
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_additive_attention_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'additive_attention' (of type AdditiveAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [10]:
nmt_additive_attention_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

nmt_additive_attention_model.fit((X_train, X_train_dec), Y_train,
                                epochs=10,
                                validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 167s 51ms/step - accuracy: 0.8579 - loss: 1.0600 - val_accuracy: 0.8584 - val_loss: 0.9561
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 159s 51ms/step - accuracy: 0.8583 - loss: 0.8921 - val_accuracy: 0.8584 - val_loss: 0.8504
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 160s 51ms/step - accuracy: 0.8583 - loss: 0.8295 - val_accuracy: 0.8584 - val_loss: 0.8216
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 160s 51ms/step - accuracy: 0.8583 - loss: 0.8059 - val_accuracy: 0.8584 - val_loss: 0.8099
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 160s 51ms/step - accuracy: 0.8583 - loss: 0.7926 - val_accuracy: 0.8583 - val_loss: 0.8029
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 202s 51ms/step - accuracy: 0.8583 - loss: 0.7834 - val_accuracy: 0.8584 - val_loss: 0.7995
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 159s 51ms/step - accuracy: 0.8583 - loss: 0.7763 - val_accuracy: 0.8582 - val_loss: 0.7976
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 159s 51ms/step - accuracy: 

## *Test Attention with Custom Model

In [ ]:
reload(tf_model)
from tf_model import *

vocab_size = 300
max_length = 50

text_vec_layer_en = TextVectorization(vocab_size, output_sequence_length=max_length)
text_vec_layer_es = TextVectorization(vocab_size, output_sequence_length=max_length)

text_vec_layer_en.adapt(sentences_en[:1000])
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es[:1000]])


X_train = tf.constant(sentences_en[:100])
X_valid = tf.constant(sentences_en[-500:])

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[-500:]])

Y_train = text_vec_layer_es.call([f"{s} endofseq" for s in sentences_es[:100]])
Y_valid = text_vec_layer_es.call([f"{s} endofseq" for s in sentences_es[-500:]])

In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = Input([])
decoder_inputs = Input([])

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = LSTM(512, return_sequences=True)
encoder_outputs = encoder(encoder_embeddings)

# DECODER
decoder = LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings)

# Attention
attention_layer = Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# OUTPUT
output_layer = Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_model = Model(inputs=[encoder_inputs, decoder_inputs], outputs=Y_proba)

In [ ]:
nmt_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

nmt_model.fit((X_train, X_train_dec), Y_train,
              epochs=10,
              validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10


  0%|          | 0/4 [00:00<?, ?it/s]

Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `expo

100%|██████████| 4/4 [00:19<00:00,  4.95s/it]


    - loss: 0.946 - accuracy: 0.405
    - val_loss: 0.937 - val_accuracy: 0.859
    - Learning Rate: 0.001
Epoch 2/10


...

    - loss: 0.419 - accuracy: 0.863
    - val_loss: 0.285 - val_accuracy: 0.860
    - Learning Rate: 0.001
Epoch 9/10


100%|██████████| 4/4 [00:05<00:00,  1.46s/it]


    - loss: 0.270 - accuracy: 0.864
    - val_loss: 0.262 - val_accuracy: 0.860
    - Learning Rate: 0.001
Epoch 10/10


100%|██████████| 4/4 [00:05<00:00,  1.49s/it]


    - loss: 0.237 - accuracy: 0.872
    - val_loss: 0.253 - val_accuracy: 0.860
    - Learning Rate: 0.001


In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = Input([])
decoder_inputs = Input([])

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = LSTM(512, return_sequences=True)
encoder_outputs = encoder(encoder_embeddings)

# DECODER
decoder = LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings)

# AdditiveAttention
attention_layer = AdditiveAttention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# OUTPUT
output_layer = Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_model = Model(inputs=[encoder_inputs, decoder_inputs], outputs=Y_proba)

In [ ]:
nmt_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

nmt_model.fit((X_train, X_train_dec), Y_train,
              epochs=5,
              validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/5


  0%|          | 0/4 [00:00<?, ?it/s]

Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `expo

W0000 00:00:1772308749.465542    4282 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" vendor: "GenuineIntel" model: "101" frequency: 2294 num_cores: 4 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 32768 l2_cache_size: 262144 l3_cache_size: 3145728 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
100%|██████████| 4/4 [00:21<00:00,  5.45s/it]


    - loss: 0.945 - accuracy: 0.438
    - val_loss: 0.935 - val_accuracy: 0.860
    - Learning Rate: 0.001
Epoch 2/5


...

    - loss: 0.910 - accuracy: 0.870
    - val_loss: 0.893 - val_accuracy: 0.860
    - Learning Rate: 0.001
Epoch 4/5


100%|██████████| 4/4 [00:10<00:00,  2.51s/it]


    - loss: 0.887 - accuracy: 0.864
    - val_loss: 0.858 - val_accuracy: 0.860
    - Learning Rate: 0.001
Epoch 5/5


100%|██████████| 4/4 [00:09<00:00,  2.31s/it]


    - loss: 0.848 - accuracy: 0.869
    - val_loss: 0.796 - val_accuracy: 0.860
    - Learning Rate: 0.001


## Task 3
да се тества примера с учебника за модел който ползва attention.

    да се сравнят резултатите с предишните имплементации.
    да се имплементира с номален RNN и да се види има ли смисъл от bidirectional RNN когато ползвме attention
    да се премахне напълно RNN-a и на attention модела да се подават направо embedding-ите


### Bidirectional

In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True, return_state=True))
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

encoder_state = [tf.keras.layers.concatenate(encoder_state[::2], axis=-1),  # short-term (0 & 2)
                 tf.keras.layers.concatenate(encoder_state[1::2], axis=-1)] # long-term (1 & 3)

# DECODER
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

# Attention
attention_layer = tf.keras.layers.Attention(score_mode='dot')
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_attention_bidirectional_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
nmt_attention_bidirectional_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

callbacks = [tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/ml_models/nmt_attention_bidirectional_model.keras')]
nmt_attention_bidirectional_model.fit((X_train, X_train_dec), Y_train,
                                    epochs=10,
                                    validation_data=((X_valid, X_valid_dec), Y_valid),
                                    callbacks=callbacks)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 97s 29ms/step - accuracy: 0.4476 - loss: 2.8777 - val_accuracy: 0.5768 - val_loss: 1.9220
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 86s 27ms/step - accuracy: 0.6302 - loss: 1.6126 - val_accuracy: 0.6588 - val_loss: 1.4642
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 139s 26ms/step - accuracy: 0.6850 - loss: 1.3114 - val_accuracy: 0.6847 - val_loss: 1.3340
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 26ms/step - accuracy: 0.7140 - loss: 1.1577 - val_accuracy: 0.6950 - val_loss: 1.2768
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 26ms/step - accuracy: 0.7357 - loss: 1.0477 - val_accuracy: 0.6996 - val_loss: 1.2609
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 26ms/step - accuracy: 0.7546 - loss: 0.9558 - val_accuracy: 0.7003 - val_loss: 1.2643
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 26ms/step - accuracy: 0.7699 - loss: 0.8790 - val_accuracy: 0.7047 - val_loss: 1.2682
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 89s 29ms/step - accuracy: 0.7840 

In [ ]:
loss, nmt_attention_bidirectional_acc = nmt_attention_bidirectional_model.evaluate((X_valid, X_valid_dec), Y_valid)

593/593 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.7012 - loss: 1.3414


In [ ]:
nmt_attention_bidirectional_acc

0.7012085318565369

In [ ]:
translate(nmt_attention_bidirectional_model, "I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 389ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


'me gusta la fútbol y también voy a la playa'

### Without Bidirectional

In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.LSTM(512, return_sequences=True, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

# DECODER
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

# Attention
attention_layer = tf.keras.layers.Attention(score_mode='dot')
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_attention_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

In [ ]:
nmt_attention_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

callbacks = [tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/ml_models/nmt_attention_model.keras')]
nmt_attention_model.fit((X_train, X_train_dec), Y_train,
                        epochs=10,
                        validation_data=((X_valid, X_valid_dec), Y_valid),
                        callbacks=callbacks)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 88s 27ms/step - accuracy: 0.3735 - loss: 3.4456 - val_accuracy: 0.4635 - val_loss: 2.6465
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.5285 - loss: 2.2102 - val_accuracy: 0.5808 - val_loss: 1.8748
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.6203 - loss: 1.6572 - val_accuracy: 0.6348 - val_loss: 1.5855
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 25ms/step - accuracy: 0.6641 - loss: 1.4155 - val_accuracy: 0.6550 - val_loss: 1.4691
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.6919 - loss: 1.2664 - val_accuracy: 0.6694 - val_loss: 1.4130
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.7146 - loss: 1.1517 - val_accuracy: 0.6736 - val_loss: 1.3946
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.7334 - loss: 1.0559 - val_accuracy: 0.6767 - val_loss: 1.3907
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.7500 -

In [ ]:
loss, nmt_attention_acc = nmt_attention_model.evaluate((X_valid, X_valid_dec), Y_valid)

593/593 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6795 - loss: 1.4366


In [ ]:
translate(nmt_attention_model, "I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


'me gusta el me [UNK] a la playa y también a la playa'

### Without RNNs

In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# Attention
attention_layer = tf.keras.layers.Attention(score_mode='dot')
attention_outputs = attention_layer([decoder_embeddings, encoder_embeddings])

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

nmt_attention_noRNNs_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

In [ ]:
nmt_attention_noRNNs_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

callbacks = [tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/ml_models/nmt_attention_noRNNs_model.keras')]
nmt_attention_noRNNs_model.fit((X_train, X_train_dec), Y_train,
                                epochs=10,
                                validation_data=((X_valid, X_valid_dec), Y_valid),
                                callbacks=callbacks)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - accuracy: 0.3108 - loss: 3.7207 - val_accuracy: 0.3951 - val_loss: 2.9239
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 53s 17ms/step - accuracy: 0.4262 - loss: 2.6227 - val_accuracy: 0.4405 - val_loss: 2.4648
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 68s 12ms/step - accuracy: 0.4560 - loss: 2.3377 - val_accuracy: 0.4538 - val_loss: 2.3268
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - accuracy: 0.4690 - loss: 2.2182 - val_accuracy: 0.4610 - val_loss: 2.2609
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - accuracy: 0.4782 - loss: 2.1478 - val_accuracy: 0.4665 - val_loss: 2.2245
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 13ms/step - accuracy: 0.4851 - loss: 2.1003 - val_accuracy: 0.4701 - val_loss: 2.2039
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 44s 14ms/step - accuracy: 0.4904 - loss: 2.0656 - val_accuracy: 0.4721 - val_loss: 2.1915
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 55s 18ms/step - accuracy: 0.4939 -

In [ ]:
loss, nmt_attention_noRNNs_acc = nmt_attention_noRNNs_model.evaluate((X_valid, X_valid_dec), Y_valid)

593/593 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.4758 - loss: 2.1700


In [ ]:
translate(nmt_attention_noRNNs_model, "I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━

'me gusta ir el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol y también [UNK] el fútbol'

### Results

In [ ]:
pd.DataFrame([
    0.0965,
    0.0945,
    nmt_attention_bidirectional_acc,
    nmt_attention_acc,
    nmt_attention_noRNNs_acc
], index=[
    'NMT NO Attention NO Bidirectional',
    'Bidirectional NO Attention',
    'Bidirectional',
    'Without Bidirectional',
    'Without RNNs'
], columns=['Valid Accuracy'])

,Valid Accuracy
NMT NO Attention NO Bidirectional,0.096500
Bidirectional NO Attention,0.094500
Bidirectional,0.701209
Without Bidirectional,0.679529
Without RNNs,0.475763
